# Credit Card Fraud Detection

IEEE-CIS Fraud Detection using SMOTE, XGBoost, decision-threshold tuning and feature importance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

TRANSACTION_PATH='data/train_transaction.csv'
IDENTITY_PATH='data/train_identity.csv'
tx=pd.read_csv(TRANSACTION_PATH)
identity=pd.read_csv(IDENTITY_PATH)
df=tx.merge(identity,on='TransactionID',how='left')
print(df.shape)

In [ ]:
y=df['isFraud'].astype(int)
X=df.drop(columns=['isFraud'])
# Keep numeric predictors for a practical, reproducible SMOTE pipeline.
X=X.select_dtypes(include=np.number)
X=X.replace([np.inf,-np.inf],np.nan)
X_train,X_valid,y_train,y_valid=train_test_split(X,y,test_size=0.20,stratify=y,random_state=42)
print('Original training class counts:', y_train.value_counts().to_dict())

imputer=SimpleImputer(strategy='median')
X_train_imp=imputer.fit_transform(X_train)
X_valid_imp=imputer.transform(X_valid)

In [ ]:
# SMOTE is applied only to the training split to avoid validation leakage.
# For memory-constrained machines, use a stratified subset before this cell.
smote=SMOTE(random_state=42)
X_res,y_res=smote.fit_resample(X_train_imp,y_train)
print('After SMOTE:', pd.Series(y_res).value_counts().to_dict())

In [ ]:
model=XGBClassifier(n_estimators=250,max_depth=6,learning_rate=0.08,subsample=0.8,colsample_bytree=0.8,eval_metric='logloss',random_state=42,n_jobs=2)
model.fit(X_res,y_res)
proba=model.predict_proba(X_valid_imp)[:,1]
print(f'ROC-AUC: {roc_auc_score(y_valid,proba):.4f}')

In [ ]:
thresholds=np.arange(0.10,0.91,0.05)
rows=[]
for t in thresholds:
    pred=(proba>=t).astype(int)
    p,r,f,_=precision_recall_fscore_support(y_valid,pred,average='binary',zero_division=0)
    rows.append((t,p,r,f))
scores=pd.DataFrame(rows,columns=['threshold','precision','recall','f1'])
best=scores.loc[scores['f1'].idxmax()]
print(scores.to_string(index=False))
print('Selected threshold:', float(best['threshold']))
final_pred=(proba>=best['threshold']).astype(int)
print(classification_report(y_valid,final_pred,digits=4))
print('Confusion matrix:\n',confusion_matrix(y_valid,final_pred))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_valid,final_pred)
plt.title('Fraud Detection - Tuned Threshold')
plt.show()

importance=pd.Series(model.feature_importances_,index=X.columns).sort_values(ascending=False).head(20)
importance.sort_values().plot(kind='barh',figsize=(8,6))
plt.title('Top 20 XGBoost Feature Importances')
plt.xlabel('Importance')
plt.show()

## Interpretation

SMOTE balances the training classes by synthesizing minority-class examples. Threshold tuning changes the precision/recall trade-off without retraining the model. Feature importance identifies variables that contribute most strongly to the fitted XGBoost model; it does not by itself establish causality.